# Optimální dýchání: frekvence a dechový objem

## Zadání slovy

> Musíte provětrat 4,2 litru alveolů za minutu. Můžete dýchat rychle a mělce,
> nebo pomalu a zhluboka — a obojí ten požadavek splní.
>
> **Které z toho stojí dýchací svaly nejmíň?**

Hluboký nádech je drahý, protože se musí roztáhnout plíce: elastická práce roste
s druhou mocninou dechového objemu. Rychlé dýchání je drahé taky, protože vzduch
musí proletět dýchacími cestami rychleji a odpor se platí druhou mocninou
průtoku. Kdyby šlo jen o tohle, nejlevnější by bylo nedýchat vůbec. Jediná věc,
která tomu brání, je požadavek na ventilaci — a proto optimum leží **na jeho
hranici**, ne uvnitř.

## Formulace

Proměnné jsou dvě: dechová frekvence $f$ [1/min] a dechový objem $V_T$ [l].
Model je Otisův, Fennův a Rahnův z roku 1950.

$$
\begin{aligned}
\text{minimize}_{f,\,V_T}\quad & \dot W = \tfrac12 E\,V_T^2 f
  + \frac{\pi^2}{120}R\,f^2 V_T^2
  && \text{vykon dychacich svalu (cmH}_2\text{O}\cdot\text{l/min)}\\
\text{subject to}\quad & f\,(V_T - V_D)\ \ge\ \dot V_A
  && \text{alveolarni ventilace (l/min)}\\
& 4 \le f \le 60,\qquad V_D < V_T \le 2{,}5
  && \text{frekvence (1/min), objem (l)}
\end{aligned}
$$

Data: $E = 10$ cmH₂O/l (elastance plic), $R = 2$ cmH₂O·s/l (odpor dýchacích
cest), $V_D = 0{,}15$ l (mrtvý prostor), $\dot V_A = 4{,}2$ l/min. Tři řádky
modelu:

- **elastická složka:** práce na roztažení plic je za jeden dech
  $\tfrac12 E V_T^2$, za minutu tedy $\tfrac12 E V_T^2 f$ — roste s hloubkou;
- **odporová složka:** při sinusovém průtoku je střední hodnota $\dot V^2$ rovna
  $(\pi f V_T)^2/2$; po převodu $R$ ze sekund na minuty vyjde
  $\tfrac{\pi^2}{120}R f^2 V_T^2$ — roste s rychlostí;
- **mrtvý prostor:** z každého dechu se k alveolům dostane jen $V_T - V_D$,
  protože 0,15 l zůstane v průdušnici a průduškách. Proto v omezení není
  $f\,V_T$, ale $f\,(V_T - V_D)$.

Takhle zapsaná úloha konvexní **není** — v účelové funkci je součin $f^2V_T^2$.
Přípustná oblast sama konvexní je (leží nad grafem konvexní funkce $V_D + \dot V_A/f$),
ale omezení $f\,v \ge \dot V_A$ je součin dvou proměnných a v tomhle tvaru ho DCP
nevezme. Po substituci $V_T = V_D + v$ jsou ale
všechny členy účelové funkce i omezení **monomy s kladnými koeficienty**, tedy
posynomy, a to je přesně definice **geometrického programu**. CVXPY takové úlohy
umí: proměnné se založí jako `pos=True` a řeší se s `gp=True`. Uvnitř se to
zlogaritmuje a stane se z toho konvexní úloha — my to zapíšeme tak, jak stojí
v modelu.

## Od zadání ke kódu

| v zadání | v kódu |
|---|---|
| $f > 0$, $v = V_T - V_D > 0$ | `f = cp.Variable(pos=True)`, `v = cp.Variable(pos=True)` |
| $V_T = V_D + v$ | `V_T = v + V_D` |
| $\tfrac12 E V_T^2 f + \tfrac{\pi^2}{120}R f^2V_T^2$ | `0.5 * E * cp.square(V_T) * f + K * R * cp.square(f) * cp.square(V_T)` |
| $f\,(V_T - V_D) \ge \dot V_A$ | `f * v >= VA` |
| meze $f$ a $V_T$ | `f >= 4, f <= 60, v <= 2.5 - V_D` |
| „je to geometrický program“ | `uloha.is_dgp()`, `uloha.solve(gp=True)` |

Posuvníky odpovídají nemocem: **fibróza** je tuhá plíce, tedy velké $E$;
**CHOPN** je ucpaná dýchací cesta, tedy velké $R$.

Plná verze téhle ukázky, ze které notebook vychází, je ve skriptu [`kod/ukazka_dychani.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C1/kod/ukazka_dychani.py) v repozitáři předmětu.

In [ ]:
try:
    import cvxpy as cp
except ImportError:
    %pip install -q cvxpy
    import cvxpy as cp

In [ ]:
import cvxpy as cp
import matplotlib.pyplot as plt
import numpy as np

## Model a řešení

In [ ]:
E = 10.0    # @param {type:"slider", min:5.0, max:30.0, step:0.5}
R = 2.0     # @param {type:"slider", min:0.5, max:15.0, step:0.5}
V_D = 0.15  # @param {type:"slider", min:0.0, max:0.40, step:0.01}
VA = 4.2    # @param {type:"number"}

K = np.pi**2 / 120  # převede odpor R ze sekund na minuty

# substituce V_T = V_D + v, v > 0: pak je všechno posynom v (f, v)
f = cp.Variable(pos=True)
v = cp.Variable(pos=True)
V_T = v + V_D
W = 0.5 * E * cp.square(V_T) * f + K * R * cp.square(f) * cp.square(V_T)
omezeni = [f * v >= VA, f >= 4, f <= 60, v <= 2.5 - V_D]
uloha = cp.Problem(cp.Minimize(W), omezeni)

print(f"je to geometrický program (DGP)?  {uloha.is_dgp()}")
print(f"je to běžná konvexní úloha (DCP)? {uloha.is_dcp()}")

uloha.solve(gp=True)
f_opt, vt_opt, w_opt = float(f.value), float(v.value) + V_D, float(uloha.value)

print(f"\nstav řešení: {uloha.status}  ({uloha.solver_stats.solver_name}, "
      f"{1e3 * uloha.solver_stats.solve_time:.2f} ms)")
print(f"optimum:  f* = {f_opt:.4f} /min,  V_T* = {vt_opt:.5f} l")
print(f"alveolární ventilace {f_opt * (vt_opt - V_D):.6f} l/min (požadavek {VA})")
print(f"výkon W* = {w_opt:.4f} cmH2O*l/min = {w_opt * 0.0980665:.4f} J/min")

## Kontrola, která umí selhat

Účelová funkce je ve $V_T$ rostoucí, takže při každé frekvenci je nejlevnější
**nejmenší přípustný** objem — omezení tedy musí být aktivní. Dosazením
$f = \dot V_A/v$ pak zmizí jedna proměnná a podmínka
$\mathrm d\dot W/\mathrm dv = 0$ se po vykrácení kladného činitele $(v + V_D)$
zredukuje na **kvadratickou rovnici**

$$a\,v^2 - a\,V_D\,v - 2b\,V_D = 0,\qquad
  a = \tfrac12 E\dot V_A,\quad b = \tfrac{\pi^2}{120}R\dot V_A^2 .$$

Solver o té rovnici nic neví, takže je to nezávislé měřítko. A kontrola má smysl
jen tehdy, když umí spadnout — pustíme ji proto i na podvržené řešení
$f = 12$/min, $V_T = 0{,}50$ l, které je přípustné na šest desetinných míst
a je to učebnicová klidová hodnota.

In [ ]:
a = 0.5 * E * VA
b = K * R * VA**2
koreny = np.roots([a, -a * V_D, -2 * b * V_D])
v_an = float(max(koreny[np.isreal(koreny)].real))
f_an, vt_an = VA / v_an, v_an + V_D
print(f"tužkou:  f = {f_an:.9f} /min,  V_T = {vt_an:.9f} l")
print(f"CVXPY:   f = {f_opt:.9f} /min,  V_T = {vt_opt:.9f} l")
assert max(abs(f_opt - f_an), abs(vt_opt - vt_an)) < 1e-3, "optimum nesedí na kořen kvadratiky"

# kontrola umí spadnout: podvržené řešení je přípustné, ale optimum to není
ff, vv = 12.0, 0.50
print(f"\npodvržené f = {ff:.0f} /min, V_T = {vv:.2f} l:"
      f" ventilace {ff * (vv - V_D):.6f} l/min, tedy přípustné,")
print(f"ale od optima je daleko ({max(abs(ff - f_an), abs(vv - vt_an)):.3f}) —"
      f" tam kontrola spadne. Samotná přípustnost by ho propustila.")

## Obrázek

Dvě proměnné jsou poslední počet, u kterého se do jednoho obrázku vejde
**přípustná oblast i krajina účelové funkce**. Vrstevnice se dá vyjádřit
uzavřeně — účelová funkce je ve $V_T$ kvadratická, takže stačí odmocnit.

In [ ]:
fs = np.linspace(5, 40, 400)
hranice = V_D + VA / fs


def vrstevnice(ff, w):
    return np.sqrt(w / (0.5 * E * ff + K * R * ff**2))


plt.figure(figsize=(7.5, 5))
plt.fill_between(fs, hranice, 1.05, color="#e6f2e9")
plt.fill_between(fs, 0, hranice, color="#e4e4ea")
plt.plot(fs, hranice, color="#1f4e79", lw=3, label=f"$f\\,(V_T - V_D) = {VA}$  (hranice)")
for w in (0.68 * w_opt, 1.9 * w_opt):  # jedna vrstevnice pod optimem, jedna nad
    plt.plot(fs, vrstevnice(fs, w), color="#c0392b", lw=1.5, ls=":")
plt.plot(fs, vrstevnice(fs, w_opt), color="#c0392b", lw=2.2, ls="--",
         label=f"vrstevnice $\\dot W$ = {w_opt:.1f} se hranice dotýká")
plt.plot(f_opt, vt_opt, "o", ms=12, color="#1a7f37",
         label=f"optimum: {f_opt:.1f}/min, {vt_opt:.2f} l")
plt.xlim(5, 40)
plt.ylim(0, 1.05)
plt.xlabel("dechová frekvence $f$ [1/min]")
plt.ylabel("dechový objem $V_T$ [l]")
plt.title("Optimum leží na hranici a vrstevnice se jí dotýká")
plt.legend(loc="upper right")
plt.show()

## Na co se zeptat kódu

1. **Fibróza.** Nastavte $E = 25$ (tuhé plíce). Kam se optimum posune po hranici
   a jak se to jmenuje u lůžka?
2. **CHOPN.** Vraťte $E = 10$ a nastavte $R = 10$. Optimum sjede na opačnou
   stranu — proč?
3. **Mrtvý prostor.** Nastavte $V_D = 0$. Kde skončí solver a proč tam vnitřní
   optimum neexistuje? (Nápověda: při $V_D = 0$ je $f^2V_T^2 = \dot V_A^2$
   konstantní.)
4. **Špatná účelová funkce.** Minimalizujte místo výkonu jen elastickou složku.
   Solver vrátí číslo i tak — jak by se na chybu přišlo bez znalosti správné
   odpovědi?